# Eksperyment: Walidacja walk-forward (wiele sezonów) + test istotności

## Cel
Sprawdzić UCZCIWIE, czy cechy wzbogacone (`surface_speed` + `fatigue`) faktycznie poprawiają model,
czy tylko **wyglądały** dobrze na jednym sezonie. Pojedynczy test set (np. 2024, ~590 meczów) ma
przedział ufności ~ +/-4 p.p., więc zysk +1-2 p.p. może być czystym szumem.

## Metoda (leakage-safe, parowana)
- **Walk-forward** przez 6 sezonów docelowych (2020-2025). Dla każdego roku `Y` osobno:
  - historia = sezony `2001..Y-1` (Sackmann), nigdy z przyszłości,
  - baseline (RF, tuned HP) trenowany na 60% `Y`, testowany na 20% `Y`,
  - model wzbogacony na **dokładnie tych samych** meczach (czysta ablacja -- zmieniamy tylko cechy).
- Zbieramy poprawność **mecz po meczu** dla OBU modeli -> test **parowany** (te same mecze).
- Raport: delta per sezon, delta **pooled** (wszystkie mecze razem) + orientacyjne CI, oraz
  **McNemar** (parowany test istotności na meczach, gdzie modele się różnią).
- Wiele niezależnych test setów -> węższe CI + sprawdzenie, czy delta jest **spójna** między latami.

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path("../src").resolve()))

## 1. Reuse modułu walk-forward
Importujemy publiczne funkcje z `tennis_model_walkforward.py`. Kluczowe:
- `run_baseline_for_year(Y)` -- uruchamia `tennis_model.py` (cicho, runpy) dla roku `Y` i zwraca jego
  namespace: chronologiczne splity, tuned hiperparametry, funkcje pomocnicze i wynik baseline.
- `enriched_eval_for_year(ns, Y)` -- na splitach z `ns` buduje model wzbogacony i zwraca parowane
  ewaluacje (per-mecz poprawność) obu modeli + ich match accuracy.
- `mcnemar(b, c)` / `wilson_delta_ci(p, n)` -- testy istotności.

Zerkamy też na listę sezonów i zestaw nowych cech.

In [2]:
import os
import numpy as np
import pandas as pd

from tennis_model_walkforward import (
    run_baseline_for_year, enriched_eval_for_year,
    mcnemar, wilson_delta_ci,
    TARGET_YEARS, HISTORY_START_YEAR,
    SPEED_FEATURES, FATIGUE_FEATURES, NEW_FEATURES,
)

print(f"Sezony docelowe (walk-forward): {TARGET_YEARS}")
print(f"Historia startuje od:           {HISTORY_START_YEAR}")
print(f"\nNowe cechy ({len(NEW_FEATURES)}):")
print(f"  surface_speed ({len(SPEED_FEATURES)}): {SPEED_FEATURES}")
print(f"  fatigue       ({len(FATIGUE_FEATURES)}): {FATIGUE_FEATURES}")

Sezony docelowe (walk-forward): [2020, 2021, 2022, 2023, 2024, 2025]
Historia startuje od:           2001

Nowe cechy (9):
  surface_speed (3): ['court_pace_index', 'ace_speed_diff', 'first_won_speed_diff']
  fatigue       (6): ['p1_rest_days', 'p2_rest_days', 'rest_days_diff', 'p1_tourney_minutes', 'p2_tourney_minutes', 'tourney_minutes_diff']


## 2. Demo jednego sezonu (2025)
Zanim puścimy pełną pętlę, pokażmy mechanikę na JEDNYM sezonie. To właśnie taki pojedynczy wynik
potrafił kiedyś "skłamać" (+2 p.p.). Liczymy baseline 2025, dokładamy cechy wzbogacone na tych samych
meczach i porównujemy match accuracy. Zwracana ramka ewaluacji ma jeden wiersz na mecz z kolumnami
`match_id` i `correct_prediction` -- to one posłużą do parowania.

In [3]:
DEMO = 2025
ns_demo = run_baseline_for_year(DEMO)                 # runpy tennis_model.py (cicho)
print(f"Baseline {DEMO}: val={ns_demo['val_acc']:.4f}  test={ns_demo['test_acc']:.4f}  "
      f"match={float(ns_demo['match_accuracy']):.4f}")
print(f"Splity {DEMO}: train={len(ns_demo['df_train_raw'])}  val={len(ns_demo['df_val_raw'])}  "
      f"test={len(ns_demo['df_test_raw'])}")

base_eval, enr_eval, base_m, enr_m = enriched_eval_for_year(ns_demo, DEMO)
print(f"\nbaseline match = {base_m:.4f}")
print(f"enriched match = {enr_m:.4f}   (delta {enr_m - base_m:+.4f})")

print("\nPrzykład parowanej ewaluacji (pierwsze 6 meczów test setu):")
demo_merged = base_eval.merge(enr_eval, on="match_id", suffixes=("_base", "_enr"))
print(demo_merged.head(6).to_string(index=False))

Baseline 2025: val=0.6106  test=0.6604  match=0.6566
Splity 2025: train=1588  val=529  test=530



baseline match = 0.6566
enriched match = 0.6585   (delta +0.0019)

Przykład parowanej ewaluacji (pierwsze 6 meczów test setu):
 match_id  correct_prediction_base  correct_prediction_enr
      486                    False                   False
      156                    False                   False
       35                     True                    True
      514                    False                   False
      358                    False                   False
      154                     True                    True


## 3. Pełna pętla walk-forward 2020-2025
Powtarzamy to samo dla KAŻDEGO sezonu osobno i zbieramy pary `(baseline poprawny, enriched poprawny)`
ze wszystkich meczów testowych. To długi bieg -- 6x pełny baseline (rok 2025 reużywamy z demo).
Drukujemy wynik per rok na bieżąco, żeby widzieć, jak niespójna potrafi być delta między sezonami.

In [4]:
rows, all_pairs = [], []
ns_cache = {DEMO: ns_demo}                            # nie liczmy 2025 drugi raz
for year in TARGET_YEARS:
    ns = ns_cache.get(year) or run_baseline_for_year(year)
    base_eval, enr_eval, base_m, enr_m = enriched_eval_for_year(ns, year)
    merged = base_eval.merge(enr_eval, on="match_id", suffixes=("_base", "_enr"))
    for _, r in merged.iterrows():
        all_pairs.append((bool(r["correct_prediction_base"]), bool(r["correct_prediction_enr"])))
    rows.append({"year": year, "n_test": len(merged),
                 "baseline": base_m, "enriched": enr_m, "delta": enr_m - base_m})
    print(f"  {year}: baseline={base_m:.4f}  enriched={enr_m:.4f}  "
          f"delta={enr_m - base_m:+.4f}  (n={len(merged)})", flush=True)
os.environ.pop("TENNIS_TARGET_YEAR", None)

df = pd.DataFrame(rows)
print("\nDelta per sezon:")
print(df.to_string(index=False, float_format=lambda x: f"{x:.4f}"))
print(f"\nDelta dodatnia w {int((df['delta'] > 0).sum())}/{len(df)} sezonach")

  2020: baseline=0.6176  enriched=0.6140  delta=-0.0037  (n=272)


  2021: baseline=0.6667  enriched=0.6667  delta=+0.0000  (n=522)


  2022: baseline=0.6728  enriched=0.6764  delta=+0.0037  (n=547)


  2023: baseline=0.6346  enriched=0.6346  delta=+0.0000  (n=561)


  2024: baseline=0.6186  enriched=0.6254  delta=+0.0068  (n=590)


  2025: baseline=0.6566  enriched=0.6585  delta=+0.0019  (n=530)



Delta per sezon:
 year  n_test  baseline  enriched   delta
 2020     272    0.6176    0.6140 -0.0037
 2021     522    0.6667    0.6667  0.0000
 2022     547    0.6728    0.6764  0.0037
 2023     561    0.6346    0.6346  0.0000
 2024     590    0.6186    0.6254  0.0068
 2025     530    0.6566    0.6585  0.0019

Delta dodatnia w 3/6 sezonach


## 4. Pooled accuracy + McNemar (parowany test istotności)
Łączymy wszystkie mecze ze wszystkich sezonów w jeden duży zbiór (N ~ 3000) -> ciaśniejsze CI niż
pojedynczy sezon. Liczymy pooled deltę oraz **McNemar**: `b` = baseline trafił, enriched nie;
`c` = odwrotnie. Istotność na korzyść enriched wymaga `p < 0.05` ORAZ `c > b`.

In [5]:
pairs = np.array(all_pairs)
base_correct, enr_correct = pairs[:, 0], pairs[:, 1]
N = len(pairs)
pooled_base, pooled_enr = float(base_correct.mean()), float(enr_correct.mean())
pooled_delta = pooled_enr - pooled_base

b = int(np.sum(base_correct & ~enr_correct))
c = int(np.sum(~base_correct & enr_correct))
z, pval = mcnemar(b, c)

print("=" * 72)
print("WALK-FORWARD: baseline vs enriched (surface_speed + fatigue)")
print("=" * 72)
print(df.to_string(index=False, float_format=lambda x: f"{x:.4f}"))
print("-" * 72)
print(f"POOLED ({N} meczów): baseline={pooled_base:.4f}  enriched={pooled_enr:.4f}  "
      f"delta={pooled_delta:+.4f}")
print(f"  Orientacyjne 95% CI (pojedyncza proporcja): +/-{wilson_delta_ci(pooled_enr, N):.4f}")
print(f"  Spójność: delta dodatnia w {int((df['delta'] > 0).sum())}/{len(df)} sezonach")
print()
print(f"McNemar (parowany): b(base+,enr-)={b}  c(base-,enr+)={c}  z={z:.2f}  p(two-sided)={pval:.4f}")
if pval < 0.05 and c > b:
    print("  => Różnica ISTOTNA statystycznie (p<0.05) na korzyść enriched.")
elif pval < 0.05 and b > c:
    print("  => Różnica istotna, ale na NIEKORZYŚĆ enriched.")
else:
    print("  => Brak istotności (p>=0.05): zysk w granicach szumu.")

WALK-FORWARD: baseline vs enriched (surface_speed + fatigue)
 year  n_test  baseline  enriched   delta
 2020     272    0.6176    0.6140 -0.0037
 2021     522    0.6667    0.6667  0.0000
 2022     547    0.6728    0.6764  0.0037
 2023     561    0.6346    0.6346  0.0000
 2024     590    0.6186    0.6254  0.0068
 2025     530    0.6566    0.6585  0.0019
------------------------------------------------------------------------
POOLED (3022 meczów): baseline=0.6463  enriched=0.6482  delta=+0.0020
  Orientacyjne 95% CI (pojedyncza proporcja): +/-0.0170
  Spójność: delta dodatnia w 3/6 sezonach

McNemar (parowany): b(base+,enr-)=60  c(base-,enr+)=66  z=0.45  p(two-sided)=0.6560
  => Brak istotności (p>=0.05): zysk w granicach szumu.


## Wnioski
Walidacja przez 6 sezonów (2020–2025, ~3000 meczów), cechy prędkość kortu + zmęczenie razem. Łącznie baseline 0,6463, wersja z cechami 0,6482 — poprawa +0,20 p.p. przy McNemar p = 0,66, czyli zupełnie nieistotna. Dodatnia tylko w 3 sezonach na 6, bez stałego trendu.

To jest główna lekcja metodologiczna: pozorny zysk z jednego sezonu, który potrafił wyglądać nawet na +2 p.p., znika, gdy sprawdzić go przez wiele lat. Pojedynczy sezon (~590 meczów) ma przedział ±4 p.p., więc taki „zysk" od początku mieścił się w szumie. Cechy prędkości i zmęczenia są więc w praktyce neutralne — ani nie szkodzą, ani nie pomagają w istotny sposób. Znów **~65% jako sufit**.